# Parte 2 – API de lluvias (Open-Meteo)

**Pregunta del trabajo:** ¿El Estado declara emergencia donde más llueve?

Grupo 1, temporada 2022 (1 de enero al 31 de mayo de 2022). Aquí medimos cuánto llovió en cada departamento usando la capital como punto de referencia. En este bloque solo hacemos los pasos 1 a 3: la lista de departamentos y capitales (Wikipedia), la verificación de las 25 filas y el arreglo de las capitales con texto extra. Todavía no geocodificamos ni pedimos lluvia.

In [1]:
ANIO = 2022
INICIO = f"{ANIO}-01-01"
FIN = f"{ANIO}-05-31"

import io
import re
import time
import requests
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. Tabla de departamentos y capitales de Wikipedia

Wikipedia rechaza `pd.read_html(url)` directo (error 403). Siguiendo la pista del enunciado, primero descargamos la página con `requests.get(..., headers={"User-Agent": "Mozilla/5.0"})` y luego le pasamos el texto a `pd.read_html(io.StringIO(respuesta.text))`. Hacemos una pausa de 1 segundo después del pedido.

**Diferencia con la clase:** el profesor mostró `requests.get(url, params=..., timeout=30)` con una API y `pd.DataFrame`, pero no `pd.read_html` ni Wikipedia (esa URL no aparece en ninguna transcripción). Usamos la página `Anexo:Departamentos_del_Perú`, que es la que trae una tabla con departamento y capital.

In [2]:
URL_WIKI = "https://es.wikipedia.org/wiki/Anexo:Departamentos_del_Perú"

respuesta = requests.get(URL_WIKI, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
print("Código de respuesta:", respuesta.status_code)
time.sleep(1)  # pausa de >= 1 s con requests

tablas = pd.read_html(io.StringIO(respuesta.text))
print("Tablas encontradas en la página:", len(tablas))
for i, t in enumerate(tablas):
    print(f"  tabla {i}: {t.shape[0]} filas x {t.shape[1]} columnas | columnas: {[str(c)[:22] for c in t.columns][:6]}")

Código de respuesta: 200


Tablas encontradas en la página: 6
  tabla 0: 1 filas x 1 columnas | columnas: ['Departamentos del Perú']
  tabla 1: 24 filas x 11 columnas | columnas: ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
  tabla 2: 8 filas x 5 columnas | columnas: ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
  tabla 3: 2 filas x 11 columnas | columnas: ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
  tabla 4: 6 filas x 4 columnas | columnas: ['Provincia', 'Capital', 'Creación', 'Desaparición']
  tabla 5: 1 filas x 2 columnas | columnas: ['0', '1']


Hay varias tablas y **dos** tienen las columnas `Departamento` y `Capital`: la tabla de los departamentos actuales y una tabla de departamentos históricos que ya no existen (La Costa, Tarma, Huaylas, Tarapacá...), que además tiene la columna `Desaparición`. No podemos elegir solo por los nombres de las columnas. Elegimos la que tiene `Departamento` y `Capital` y **no** tiene `Desaparición`. Además de elegirla, comprobamos cuántas filas tiene.

In [3]:
candidatas = [i for i, t in enumerate(tablas)
              if {"Departamento", "Capital"} <= set(map(str, t.columns)) and "Desaparición" not in set(map(str, t.columns))]
print("Tablas candidatas:", candidatas)

departamentos_wiki = tablas[candidatas[0]]
print("Filas de la tabla elegida:", len(departamentos_wiki))

# Solo los NOMBRES de las columnas pasan a minúsculas; los valores quedan tal como están (mayúsculas y tildes)
departamentos_df = departamentos_wiki[["Departamento", "Capital"]].rename(columns={"Departamento": "departamento", "Capital": "capital"})
departamentos_df

Tablas candidatas: [1]
Filas de la tabla elegida: 24


,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## 2. Verificación: ¿están los 25?

La tabla final debe tener 25 filas (los 24 departamentos y el Callao). Comparamos con la lista de departamentos del enunciado (la misma de la Parte 1).

In [4]:
lista_enunciado = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                   "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                   "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                   "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                   "San Martín", "Tacna", "Tumbes", "Ucayali"]

print("Filas en la tabla de Wikipedia:", len(departamentos_df), "| esperadas:", len(lista_enunciado))
print("Falta en la tabla:", sorted(set(lista_enunciado) - set(departamentos_df["departamento"])))
print("Sobra en la tabla:", sorted(set(departamentos_df["departamento"]) - set(lista_enunciado)))

Filas en la tabla de Wikipedia: 24 | esperadas: 25
Falta en la tabla: ['Callao']
Sobra en la tabla: []


**Falta el Callao.** La tabla tiene 24 filas. Wikipedia no lo pone en la tabla de departamentos porque el Callao **no es un departamento sino una "Provincia Constitucional"** (la página lo trata junto con Lima como una de las "provincias de régimen especial"). Está en otra tabla de la misma página, la de provincias, con la fila `Callao[nota 6]`, nombre oficial "Provincia Constitucional del Callao", capital `Callao`. De ahí tomamos el Callao: lo agregamos como el departamento número 25, con su capital tal como la da esa tabla. Para el trabajo se trata como un departamento más (el INEI le da el ubigeo 07 y el enunciado lo cuenta entre los 25).

La nota entre corchetes (`[nota 6]`) no es parte del nombre: la quitamos del nombre. Es lo único que se cambia del valor.

In [5]:
tabla_provincias = [t for t in tablas if "Provincia" in set(map(str, t.columns)) and "Capital" in set(map(str, t.columns))][0]
fila_callao = tabla_provincias[tabla_provincias["Provincia"].str.startswith("Callao")]
print("Fila de Wikipedia (tabla de provincias):")
print(fila_callao[["Provincia", "Nombre oficial", "Capital"]].to_string(index=False))

nombre_callao = re.sub(r"\[.*?\]", "", fila_callao["Provincia"].iloc[0]).strip()   # "Callao[nota 6]" -> "Callao"
capital_callao = fila_callao["Capital"].iloc[0].strip()
print("\nSe agrega:", nombre_callao, "|", capital_callao)

departamentos_df = pd.concat(
    [departamentos_df, pd.DataFrame([{"departamento": nombre_callao, "capital": capital_callao}])],
    ignore_index=True)

print("\nFilas ahora:", len(departamentos_df))
print("Falta:", sorted(set(lista_enunciado) - set(departamentos_df["departamento"])))
print("Sobra:", sorted(set(departamentos_df["departamento"]) - set(lista_enunciado)))
print("¿Son exactamente los 25 del enunciado?", set(departamentos_df["departamento"]) == set(lista_enunciado) and len(departamentos_df) == 25)

Fila de Wikipedia (tabla de provincias):
     Provincia                      Nombre oficial Capital
Callao[nota 6] Provincia Constitucional del Callao  Callao

Se agrega: Callao | Callao

Filas ahora: 25
Falta: []
Sobra: []
¿Son exactamente los 25 del enunciado? True


## 3. Capitales con texto extra

El enunciado avisa que algunas celdas de `capital` traen texto extra (por ejemplo "Huacho (de facto)") y que así la API de geocodificación no las va a encontrar. Buscamos todas las capitales con paréntesis o corchetes, para no arreglar solo la que ya conocemos.

In [6]:
con_extra = departamentos_df["capital"].str.contains(r"[\(\[]", regex=True)
print("Capitales con paréntesis o corchetes:", con_extra.sum())
print(departamentos_df[con_extra].to_string())
print()
print("Otras señales de texto raro (espacios al borde, espacios dobles, números):")
print("  espacios al borde:", (departamentos_df["capital"] != departamentos_df["capital"].str.strip()).sum())
print("  espacios dobles:", departamentos_df["capital"].str.contains("  ").sum())
print("  con números:", departamentos_df["capital"].str.contains(r"\d").sum())

Capitales con paréntesis o corchetes: 1
   departamento            capital
13         Lima  Huacho (de facto)

Otras señales de texto raro (espacios al borde, espacios dobles, números):
  espacios al borde: 0
  espacios dobles: 0
  con números: 0


In [7]:
capital_original = departamentos_df["capital"].copy()

# Quitamos el texto entre paréntesis o corchetes y los espacios sobrantes
departamentos_df["capital"] = (departamentos_df["capital"]
                               .str.replace(r"\s*[\(\[].*?[\)\]]", "", regex=True)
                               .str.strip())

cambios = departamentos_df["capital"] != capital_original
print("Capitales que cambiaron:", cambios.sum())
print(pd.DataFrame({"antes": capital_original[cambios], "despues": departamentos_df.loc[cambios, "capital"]}).to_string())

Capitales que cambiaron: 1
                antes despues
13  Huacho (de facto)  Huacho


**Decisión sobre Lima.** Wikipedia dice que la capital del departamento de Lima es "Huacho (de facto)". Usamos **Huacho** (quitando solo el "(de facto)") porque es la capital del departamento según la propia tabla, y el trabajo es por departamento. Es una decisión **provisoria**: el grupo la confirma.

La alternativa es **Lima**:

| | Huacho | Lima |
|---|---|---|
| A favor | Es lo que dice la tabla para el departamento de Lima; respeta la regla "una capital por departamento" igual que los demás. | Es la ciudad más grande y conocida; es más probable que la API la encuentre sin ambigüedad y es el punto que la mayoría de la gente asocia con "la lluvia en Lima". |
| En contra | Es una ciudad pequeña del norte del departamento, lejos de Lima Metropolitana; su lluvia no representa a la ciudad donde vive la mayor parte de la población. El nombre puede costar más en la API (hay que comprobarlo en el paso 5). | La ciudad de Lima es la capital del país y de la Provincia de Lima, que Wikipedia trata como de régimen especial, **no** la capital del departamento. Cambiaríamos la regla solo para este caso. |

En ambos casos queda una limitación: una sola ciudad costera no representa el clima de todo el departamento de Lima (que incluye provincias de la sierra). Esto va a las limitaciones del análisis de la Parte 3. Mismo caso en otros departamentos, donde la ciudad más poblada no es la capital (por ejemplo Puno/Juliaca o San Martín/Tarapoto): seguimos la regla del enunciado y usamos la capital.

In [8]:
print("Filas:", len(departamentos_df), "| departamentos únicos:", departamentos_df["departamento"].nunique())
print("Capitales con paréntesis o corchetes que quedan:", departamentos_df["capital"].str.contains(r"[\(\[]", regex=True).sum())
print("Capitales vacías o nulas:", (departamentos_df["capital"].isna() | (departamentos_df["capital"] == "")).sum())
print()
departamentos_df

Filas: 25 | departamentos únicos: 25
Capitales con paréntesis o corchetes que quedan: 0
Capitales vacías o nulas: 0



,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## 4. Latitud y longitud de cada capital (API de geocodificación)

Usamos la API de geocodificación de Open-Meteo (no necesita clave). Como en la clase de APIs, un API es un link al que le pasamos parámetros: `requests.get(url, params=..., timeout=30)` y luego `.json()`. Pedimos `name=<capital>`, `count=10` y `language=es`, con una pausa de **1 segundo** entre pedidos.

Guardamos **todos** los resultados que devuelve la API por capital (hasta 10), no solo el primero, porque la API puede devolver ciudades con el mismo nombre dentro y fuera del Perú y hay que poder revisar cuál se eligió. Si una capital no tiene resultados, la clave `results` ni siquiera aparece en la respuesta, así que usamos `.get("results", [])`.

In [9]:
URL_GEO = "https://geocoding-api.open-meteo.com/v1/search"

respuestas_geo = {}   # departamento -> lista con todos los resultados de la API para su capital
errores_geo = []

for _, fila in departamentos_df.iterrows():
    try:
        r = requests.get(URL_GEO, params={"name": fila["capital"], "count": 10, "language": "es"}, timeout=30)
        r.raise_for_status()
        respuestas_geo[fila["departamento"]] = r.json().get("results", [])
    except requests.RequestException as e:
        respuestas_geo[fila["departamento"]] = []
        errores_geo.append((fila["departamento"], fila["capital"], repr(e)))
    time.sleep(1)  # pausa de >= 1 s entre pedidos a la API

print("Capitales consultadas:", len(respuestas_geo))
print("Pedidos con error:", len(errores_geo), errores_geo)
print("Capitales sin ningún resultado:", [d for d, res in respuestas_geo.items() if len(res) == 0])
print()
print("Así se ve un resultado de la API (primero de Amazonas / Chachapoyas):")
print(respuestas_geo["Amazonas"][0])

Capitales consultadas: 25
Pedidos con error: 0 []
Capitales sin ningún resultado: []

Así se ve un resultado de la API (primero de Amazonas / Chachapoyas):
{'id': 3698540, 'name': 'Chachapoyas', 'latitude': -6.23169, 'longitude': -77.86903, 'elevation': 2360.0, 'feature_code': 'PPLA', 'country_code': 'PE', 'admin1_id': 3699699, 'admin2_id': 3698538, 'admin3_id': 8350561, 'timezone': 'America/Lima', 'population': 32026, 'country_id': 3932488, 'country': 'Perú', 'admin1': 'Departamento de Amazonas', 'admin2': 'Provincia de Chachapoyas', 'admin3': 'Chachapoyas'}


## 5. Verificación: ¿cada capital quedó en el Perú y en el departamento correcto?

Para cada capital recorremos los resultados y nos quedamos con el **primero que tiene `country_code == "PE"`**. Después comparamos su `admin1` con el departamento. `admin1` no siempre viene escrito igual ("Departamento de Cusco", "Ancash" sin tilde, "Provincia Constitucional del Callao"), así que para comparar quitamos las tildes, pasamos a minúsculas y quitamos solo los prefijos de este tipo ("Departamento de", "Provincia Constitucional del", "Provincia de", "Región de"). La comparación es de igualdad exacta después de eso: no aceptamos coincidencias parciales.

La columna `coincide` solo dice si el `admin1` es el departamento; no cambiamos el resultado por nuestra cuenta cuando no coincide.

In [10]:
import unicodedata


def normalizar_admin(texto):
    """Minúsculas, sin tildes y sin prefijos como 'Departamento de', 'Provincia Constitucional del'."""
    if texto is None:
        return None
    t = unicodedata.normalize("NFD", str(texto))
    t = "".join(c for c in t if unicodedata.category(c) != "Mn").lower().strip()
    t = re.sub(r"^(departamento|provincia constitucional|provincia|region)\s+(del|de)\s+", "", t)
    return t.strip()


def primer_resultado_peru(resultados):
    """Primer resultado con country_code == 'PE' y cuántos resultados venían antes de él."""
    for posicion, res in enumerate(resultados, start=1):
        if res.get("country_code") == "PE":
            return res, posicion
    return None, None


filas = []
for _, fila in departamentos_df.iterrows():
    resultados = respuestas_geo[fila["departamento"]]
    res, posicion = primer_resultado_peru(resultados)
    admin1 = res.get("admin1") if res else None
    filas.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "admin1": admin1,
        "latitud": res.get("latitude") if res else None,
        "longitud": res.get("longitude") if res else None,
        "coincide": normalizar_admin(admin1) == normalizar_admin(fila["departamento"]) if res else False,
        "_nombre_api": res.get("name") if res else None,
        "_posicion": posicion,
        "_n_resultados": len(resultados),
        "_n_peru": sum(1 for x in resultados if x.get("country_code") == "PE"),
    })
geo_df = pd.DataFrame(filas)

print("Capitales con al menos un resultado en el Perú:", geo_df["admin1"].notna().sum(), "de", len(geo_df))
print("Capitales cuyo admin1 coincide con el departamento:", geo_df["coincide"].sum(), "de", len(geo_df))
print()
geo_df[["departamento", "capital", "admin1", "latitud", "longitud", "coincide"]]

Capitales con al menos un resultado en el Perú: 25 de 25
Capitales cuyo admin1 coincide con el departamento: 25 de 25



,departamento,capital,admin1,latitud,longitud,coincide
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,True


In [11]:
# Detalle de cómo salió la elección: nombre que devolvió la API, en qué posición estaba el primer resultado peruano y cuántos había
geo_df[["departamento", "capital", "_nombre_api", "_posicion", "_n_resultados", "_n_peru"]].rename(
    columns={"_nombre_api": "nombre_api", "_posicion": "posicion_primer_PE", "_n_resultados": "n_resultados", "_n_peru": "n_en_peru"})

,departamento,capital,nombre_api,posicion_primer_PE,n_resultados,n_en_peru
0,Amazonas,Chachapoyas,Chachapoyas,1,4,3
1,Áncash,Huaraz,Huaraz,1,1,1
2,Apurímac,Abancay,Abancay,1,1,1
3,Arequipa,Arequipa,Arequipa,1,9,7
4,Ayacucho,Ayacucho,Ayacucho,1,10,1
5,Cajamarca,Cajamarca,Cajamarca,1,10,3
6,Cusco,Cusco,Cuzco,1,5,4
7,Huancavelica,Huancavelica,Huancavelica,1,3,3
8,Huánuco,Huánuco,Huánuco,1,5,5
9,Ica,Ica,Ica,1,10,1


In [12]:
# Casos que NO se resuelven solos: no encontrada, o admin1 distinto del departamento. No se elige otra ciudad: se muestran para decidir.
problemas = geo_df[~geo_df["coincide"]]
print("Capitales con problema:", len(problemas))
for _, f in problemas.iterrows():
    print()
    print(f"{f['departamento']} / {f['capital']} -> admin1 elegido: {f['admin1']!r} (posición {f['_posicion']})")
    print("  Todos los resultados que devolvió la API:")
    for i, x in enumerate(respuestas_geo[f["departamento"]], start=1):
        print(f"   {i:>2}. {x.get('name')!r} | {x.get('country_code')} | admin1={x.get('admin1')!r} | admin2={x.get('admin2')!r} | "
              f"lat={x.get('latitude')} lon={x.get('longitude')} | población={x.get('population')}")

Capitales con problema: 0


**Cómo elegimos el resultado correcto**

1. **Primero el país.** De los hasta 10 resultados de cada capital nos quedamos con el **primero con `country_code == "PE"`**. En las 25 capitales ese resultado ya era el primero de la lista (posición 1), pero el filtro sigue siendo necesario: por ejemplo "Trujillo" devuelve 10 resultados de varios países (Venezuela, Honduras, España...), y entre los 2 del Perú hay otro Trujillo, de Loreto (posición 7), que no es el que buscamos. Si hubiéramos tomado simplemente "el resultado con más población" o "el último del Perú", habríamos podido equivocarnos.
2. **Después el departamento.** Revisamos fila por fila que `admin1` fuera el departamento. Las variantes que aparecieron fueron: `Departamento de X` (la mayoría), `Ancash` sin tilde (frente a "Áncash"), `Apurímac`, `Puno` y `Ucayali` sin prefijo y `Provincia Constitucional del Callao`. Para compararlas quitamos tildes, pasamos a minúsculas y quitamos solo esos prefijos; la comparación es de igualdad exacta, sin coincidencias parciales.
3. **Resultado:** las 25 capitales tienen un resultado en el Perú y las 25 tienen `admin1` igual al departamento (`coincide = True`). No hubo capitales sin resultado ni pedidos con error, y no tuvimos que elegir otra ciudad en ningún caso. El nombre que devolvió la API es el de la capital en 23 casos; en los otros dos solo cambia la grafía: **Cusco → "Cuzco"** y **Callao → "El Callao"**.

**Un error mío que apareció en esta revisión.** En la primera ejecución la tabla marcó `coincide = False` para La Libertad / Trujillo, aunque su `admin1` era "Departamento de La Libertad", o sea, correcto. La causa estaba en mi función de comparación: al quitar el prefijo con la expresión `(del|de la|de)`, "Departamento **de la** Libertad" perdía también el "La" que es parte del nombre y quedaba "libertad" frente a "la libertad". Quité la variante `de la` (basta con `del|de`) y volví a ejecutar: 25 de 25 coinciden. Lo importante es que el problema estaba en mi código y no en los datos: lo detecté porque la tabla mostraba el `admin1` correcto con `False`, y comprobé a mano los 10 resultados de Trujillo antes de tocar nada.

**Qué prueba y qué no prueba esta verificación.** Prueba que cada punto está en el Perú y dentro del departamento correcto, y que el nombre devuelto es el de la capital. No prueba que las coordenadas sean exactamente el centro de la ciudad ni que la capital represente la lluvia de todo el departamento (ver la nota sobre Lima y Huacho de arriba). Huacho se resolvió bien en la API (posición 1, `Departamento de Lima`), así que la alternativa "Lima" sigue siendo solo una decisión del grupo.

## 6. Lluvia diaria de cada capital (API histórica de Open-Meteo)

Para cada capital pedimos `precipitation_sum` (lluvia diaria acumulada en mm) del **2022-01-01 al 2022-05-31** a la API histórica (`archive-api.open-meteo.com`), con `timezone=America/Lima` y una pausa de **1 segundo** entre pedidos. Como en la clase de APIs: `requests.get(url, params=..., timeout=30)` y `.json()`. Las coordenadas son las que obtuvimos en el paso 5 (`geo_df`).

La respuesta trae `daily.time` (las fechas) y `daily.precipitation_sum` (un valor por día; si un día no tiene dato viene como `None`). Guardamos todo tal como llega; no cambiamos ningún valor en este paso.

In [13]:
URL_ARCHIVO = "https://archive-api.open-meteo.com/v1/archive"

lluvia_crudo = {}   # departamento -> respuesta resumida de la API
errores_lluvia = []

for _, fila in geo_df.iterrows():
    parametros = {
        "latitude": fila["latitud"],
        "longitude": fila["longitud"],
        "start_date": INICIO,
        "end_date": FIN,
        "daily": "precipitation_sum",
        "timezone": "America/Lima",
    }
    try:
        r = requests.get(URL_ARCHIVO, params=parametros, timeout=30)
        r.raise_for_status()
        datos = r.json()
        lluvia_crudo[fila["departamento"]] = {
            "fechas": datos["daily"]["time"],
            "lluvia": datos["daily"]["precipitation_sum"],
            "unidad": datos["daily_units"]["precipitation_sum"],
            "lat_api": datos["latitude"],
            "lon_api": datos["longitude"],
            "zona": datos["timezone"],
        }
    except (requests.RequestException, KeyError, ValueError) as e:
        errores_lluvia.append((fila["departamento"], fila["capital"], repr(e)))
    time.sleep(1)  # pausa de >= 1 s entre pedidos a la API

print("Capitales consultadas:", len(geo_df), "| respuestas guardadas:", len(lluvia_crudo))
print("Pedidos con error:", len(errores_lluvia))
for e in errores_lluvia:
    print("  ", e)
print()
ejemplo = lluvia_crudo["Amazonas"]
print("Ejemplo (Amazonas / Chachapoyas): unidad =", ejemplo["unidad"], "| zona =", ejemplo["zona"])
print("  primeras fechas:", ejemplo["fechas"][:5])
print("  primeros valores:", ejemplo["lluvia"][:5])

Capitales consultadas: 25 | respuestas guardadas: 25
Pedidos con error: 0

Ejemplo (Amazonas / Chachapoyas): unidad = mm | zona = America/Lima
  primeras fechas: ['2022-01-01', '2022-01-02', '2022-01-03', '2022-01-04', '2022-01-05']
  primeros valores: [3.1, 3.8, 0.9, 2.5, 1.2]


## 7. Verificación: 151 días por capital y días sin dato

Del 2022-01-01 al 2022-05-31 hay **151 días** (31 + 28 + 31 + 30 + 31). Comprobamos para cada capital: cuántos días devolvió, que la primera y la última fecha sean las pedidas, que no haya fechas repetidas, y cuántos valores vienen como `None` (sin dato).

In [14]:
filas = []
for _, fila in geo_df.iterrows():
    d = fila["departamento"]
    resp = lluvia_crudo[d]
    fechas, lluvia = resp["fechas"], resp["lluvia"]
    filas.append({
        "departamento": d,
        "capital": fila["capital"],
        "dias_devueltos": len(fechas),
        "dias_lluvia_devueltos": len(lluvia),
        "primera_fecha": fechas[0],
        "ultima_fecha": fechas[-1],
        "fechas_repetidas": len(fechas) - len(set(fechas)),
        "dias_sin_dato": sum(1 for x in lluvia if x is None),
        "unidad": resp["unidad"],
    })
verificacion_dias = pd.DataFrame(filas)

print("Capitales con 151 días:", int((verificacion_dias["dias_devueltos"] == 151).sum()), "de", len(verificacion_dias))
print("Capitales con 151 valores de lluvia:", int((verificacion_dias["dias_lluvia_devueltos"] == 151).sum()), "de", len(verificacion_dias))
print("Primera fecha distinta de", INICIO, ":", int((verificacion_dias["primera_fecha"] != INICIO).sum()), "| última fecha distinta de", FIN, ":", int((verificacion_dias["ultima_fecha"] != FIN).sum()))
print("Fechas repetidas en total:", int(verificacion_dias["fechas_repetidas"].sum()))
print("Unidades:", verificacion_dias["unidad"].unique().tolist())
print()
print("DÍAS SIN DATO (None) EN TOTAL:", int(verificacion_dias["dias_sin_dato"].sum()), "de", int(verificacion_dias["dias_lluvia_devueltos"].sum()), "valores")
print("Departamentos con al menos un día sin dato:", verificacion_dias.loc[verificacion_dias["dias_sin_dato"] > 0, "departamento"].tolist())
print("Valores no numéricos (que no sean None):", sum(1 for r in lluvia_crudo.values() for x in r["lluvia"] if x is not None and not isinstance(x, (int, float))))
print("Valores negativos:", sum(1 for r in lluvia_crudo.values() for x in r["lluvia"] if x is not None and x < 0))
print()
verificacion_dias[["departamento", "capital", "dias_devueltos", "primera_fecha", "ultima_fecha", "dias_sin_dato"]]

Capitales con 151 días: 25 de 25
Capitales con 151 valores de lluvia: 25 de 25
Primera fecha distinta de 2022-01-01 : 0 | última fecha distinta de 2022-05-31 : 0
Fechas repetidas en total: 0
Unidades: ['mm']

DÍAS SIN DATO (None) EN TOTAL: 0 de 3775 valores
Departamentos con al menos un día sin dato: []
Valores no numéricos (que no sean None): 0
Valores negativos: 0



,departamento,capital,dias_devueltos,primera_fecha,ultima_fecha,dias_sin_dato
0,Amazonas,Chachapoyas,151,2022-01-01,2022-05-31,0
1,Áncash,Huaraz,151,2022-01-01,2022-05-31,0
2,Apurímac,Abancay,151,2022-01-01,2022-05-31,0
3,Arequipa,Arequipa,151,2022-01-01,2022-05-31,0
4,Ayacucho,Ayacucho,151,2022-01-01,2022-05-31,0
5,Cajamarca,Cajamarca,151,2022-01-01,2022-05-31,0
6,Cusco,Cusco,151,2022-01-01,2022-05-31,0
7,Huancavelica,Huancavelica,151,2022-01-01,2022-05-31,0
8,Huánuco,Huánuco,151,2022-01-01,2022-05-31,0
9,Ica,Ica,151,2022-01-01,2022-05-31,0


**Qué hicimos con los días sin dato (`None`)**

**Hay 0.** Las 25 capitales devolvieron 151 días con 151 valores de lluvia cada una (25 × 151 = 3 775 valores), y **ninguno** vino como `None`: 0 días sin dato en total y en cada departamento. Tampoco hubo fechas repetidas, ni valores negativos, ni valores que no fueran números.

Por eso **no tuvimos que excluir, rellenar ni interpolar ningún día**: el 0 es el resultado de haberlo comprobado, no una suposición. Aun así, las dos funciones de cálculo están escritas para que, si en otra temporada o con otras coordenadas aparecieran `None`, no se rompan: `lluvia_total` ignora los `None` en la suma y `dias_fuertes` no los cuenta como lluvia fuerte (un día sin dato no es un día de 0 mm). Un `0.0` **sí es un dato** (un día sin lluvia): por ejemplo Tacna tiene 139 días con 0.0 y eso no es "sin dato". Esta verificación vale solo para esta temporada y estas 25 coordenadas.

## 8. Lluvia total y días de lluvia fuerte por departamento

- `lluvia_total_mm`: la **suma** de la lluvia diaria de la temporada (en mm).
- `dias_lluvia_fuerte`: el número de días con **20 mm o más** (`>= 20`).

Los días sin dato (`None`) no se suman ni cuentan como lluvia fuerte; en el paso anterior vimos cuántos hay. La suma se redondea a 1 decimal, que es la resolución de los datos, solo para quitar el ruido de coma flotante.

In [15]:
def lluvia_total(valores):
    return round(sum(x for x in valores if x is not None), 1)


def dias_fuertes(valores):
    return sum(1 for x in valores if x is not None and x >= 20)


filas = []
for _, fila in geo_df.iterrows():
    valores = lluvia_crudo[fila["departamento"]]["lluvia"]
    filas.append({
        "departamento": fila["departamento"],
        "capital": fila["capital"],
        "latitud": fila["latitud"],
        "longitud": fila["longitud"],
        "lluvia_total_mm": lluvia_total(valores),
        "dias_lluvia_fuerte": dias_fuertes(valores),
    })
lluvias_por_departamento = pd.DataFrame(filas)

# Control cruzado: el mismo cálculo con pandas
control = pd.DataFrame({d: pd.Series(r["lluvia"], dtype="float64") for d, r in lluvia_crudo.items()})
print("Control con pandas (suma y días >= 20 mm) coincide:",
      bool((control.sum().round(1).reindex(lluvias_por_departamento["departamento"]).values == lluvias_por_departamento["lluvia_total_mm"].values).all()),
      bool(((control >= 20).sum().reindex(lluvias_por_departamento["departamento"]).values == lluvias_por_departamento["dias_lluvia_fuerte"].values).all()))
print("Filas:", len(lluvias_por_departamento))
print()
lluvias_por_departamento.sort_values("lluvia_total_mm", ascending=False).reset_index(drop=True)

Control con pandas (suma y días >= 20 mm) coincide: True True
Filas: 25



,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Loreto,Iquitos,-3.74814,-73.25290,1581.8,25
1,Ucayali,Pucallpa,-8.37915,-74.55387,1288.8,20
2,Áncash,Huaraz,-9.52614,-77.52869,975.3,7
3,Junín,Huancayo,-12.06866,-75.21027,910.4,5
4,Madre de Dios,Puerto Maldonado,-12.58946,-69.19948,908.0,13
5,Huancavelica,Huancavelica,-12.78691,-74.97298,803.8,1
6,San Martín,Moyobamba,-6.03441,-76.97423,789.8,4
7,Puno,Puno,-15.84003,-70.02198,779.5,3
8,Pasco,Cerro de Pasco,-10.66577,-76.25309,774.0,0
9,Cajamarca,Cajamarca,-7.16378,-78.50027,773.3,1


In [16]:
# Para mirar si algo se ve raro: mismas cifras junto con la altitud de la capital, y el máximo diario y los días sin lluvia
extra = pd.DataFrame({
    "departamento": list(lluvia_crudo),
    "max_dia_mm": [max(x for x in r["lluvia"] if x is not None) for r in lluvia_crudo.values()],
    "dias_sin_lluvia": [sum(1 for x in r["lluvia"] if x is not None and x == 0) for r in lluvia_crudo.values()],
    "lat_api": [r["lat_api"] for r in lluvia_crudo.values()],
    "lon_api": [r["lon_api"] for r in lluvia_crudo.values()],
})
vista = lluvias_por_departamento.merge(extra, on="departamento").sort_values("lluvia_total_mm", ascending=False).reset_index(drop=True)
vista[["departamento", "capital", "lluvia_total_mm", "dias_lluvia_fuerte", "max_dia_mm", "dias_sin_lluvia", "latitud", "lat_api", "longitud", "lon_api"]]

,departamento,capital,lluvia_total_mm,dias_lluvia_fuerte,max_dia_mm,dias_sin_lluvia,latitud,lat_api,longitud,lon_api
0,Loreto,Iquitos,1581.8,25,41.1,10,-3.74814,-3.760984,-73.25290,-73.257540
1,Ucayali,Pucallpa,1288.8,20,128.8,16,-8.37915,-8.400702,-74.55387,-74.549350
2,Áncash,Huaraz,975.3,7,28.1,10,-9.52614,-9.525483,-77.52869,-77.545685
3,Junín,Huancayo,910.4,5,24.0,15,-12.06866,-12.056238,-75.21027,-75.202150
4,Madre de Dios,Puerto Maldonado,908.0,13,39.3,23,-12.58946,-12.618628,-69.19948,-69.230774
5,Huancavelica,Huancavelica,803.8,1,21.0,14,-12.78691,-12.688928,-74.97298,-74.918460
6,San Martín,Moyobamba,789.8,4,27.7,3,-6.03441,-6.010544,-76.97423,-77.014190
7,Puno,Puno,779.5,3,22.4,40,-15.84003,-15.852372,-70.02198,-70.028320
8,Pasco,Cerro de Pasco,774.0,0,15.2,5,-10.66577,-10.650263,-76.25309,-76.257720
9,Cajamarca,Cajamarca,773.3,1,22.3,10,-7.16378,-7.205624,-78.50027,-78.502530


In [17]:
# Solo lectura, sin corregir nada: ¿en qué día cae el máximo de cada capital y qué tan lejos queda el punto de la API del punto pedido?
filas = []
for d, r in lluvia_crudo.items():
    valores = r["lluvia"]
    i = max(range(len(valores)), key=lambda k: -1 if valores[k] is None else valores[k])
    filas.append({
        "departamento": d,
        "fecha_max": r["fechas"][i],
        "max_dia_mm": valores[i],
        "pct_del_total": round(100 * valores[i] / sum(x for x in valores if x is not None), 1) if sum(x for x in valores if x is not None) > 0 else None,
    })
dia_maximo = pd.DataFrame(filas).sort_values("max_dia_mm", ascending=False).reset_index(drop=True)
print("Cinco días más lluviosos de todas las capitales:")
print(dia_maximo.head(5).to_string(index=False))

print()
print("Pucallpa (Ucayali): lluvia de los días alrededor de su máximo")
r = lluvia_crudo["Ucayali"]
i = r["lluvia"].index(max(r["lluvia"]))
for k in range(max(0, i - 3), min(len(r["lluvia"]), i + 4)):
    print("  ", r["fechas"][k], r["lluvia"][k])

# Distancia aproximada entre el punto pedido y la celda que usó la API (1 grado ~ 111 km)
import math
fila_geo = geo_df.set_index("departamento")
dist = {d: math.hypot((fila_geo.loc[d, "latitud"] - r["lat_api"]) * 111, (fila_geo.loc[d, "longitud"] - r["lon_api"]) * 111 * math.cos(math.radians(fila_geo.loc[d, "latitud"])))
        for d, r in lluvia_crudo.items()}
dist = pd.Series(dist).sort_values(ascending=False)
print()
print("Distancia (km) entre la capital y el punto que usó la API: máxima", round(dist.iloc[0], 1), f"({dist.index[0]})", "| mediana", round(dist.median(), 1))

Cinco días más lluviosos de todas las capitales:
 departamento  fecha_max  max_dia_mm  pct_del_total
      Ucayali 2022-04-01       128.8           10.0
       Loreto 2022-03-27        41.1            2.6
Madre de Dios 2022-02-25        39.3            4.3
  La Libertad 2022-03-03        30.4           30.7
     Amazonas 2022-04-01        29.4            6.7

Pucallpa (Ucayali): lluvia de los días alrededor de su máximo
   2022-03-29 0.5
   2022-03-30 1.9
   2022-03-31 2.1
   2022-04-01 128.8
   2022-04-02 2.0
   2022-04-03 0.0
   2022-04-04 0.2

Distancia (km) entre la capital y el punto que usó la API: máxima 12.4 (Huancavelica) | mediana 2.9


## 9. Tabla ordenada y cosas que conviene revisar (no se corrigió nada)

La tabla de la sección 8 está ordenada de mayor a menor `lluvia_total_mm`. Lo medido:

- **Arriba:** Loreto (Iquitos) 1 581.8 mm, Ucayali (Pucallpa) 1 288.8 mm, Áncash (Huaraz) 975.3 mm, Junín (Huancayo) 910.4 mm y Madre de Dios (Puerto Maldonado) 908.0 mm.
- **Abajo:** Tacna 11.5 mm, Lima (Huacho) 13.9 mm, Moquegua 17.8 mm e Ica 25.1 mm.
- **No apareció lo que preguntabas:** ninguna capital de la costa seca queda por encima de la selva. Las dos más lluviosas son de selva (Loreto y Ucayali) y las cuatro menos lluviosas son de la costa sur y central.

**Aun así, hay cosas que conviene mirar. No las corregí:**

1. **Un día extremo en Pucallpa (Ucayali).** El 2022-04-01 la API da **128.8 mm en un solo día**, con los días vecinos entre 0.0 y 2.1 mm (31 de marzo: 2.1; 2 de abril: 2.0). Es el máximo diario de las 25 capitales y es **3 veces** el siguiente (Loreto, 41.1 mm). Representa el 10.0 % de los 1 288.8 mm de Ucayali y cuenta como uno de sus 20 `dias_lluvia_fuerte`. Con estos datos no puedo decir si fue una tormenta real o un valor atípico de los datos; no lo sé. Si se excluyera, Ucayali seguiría segundo (1 160.0 mm), pero tendría 19 días fuertes en lugar de 20.
2. **Trujillo (La Libertad).** Su máximo diario es 30.4 mm (2022-03-03) y equivale al **30.7 %** de su total de la temporada (98.9 mm); es su único día fuerte. Una sola jornada pesa casi un tercio del total.
3. **Los datos son de una celda de la cuadrícula del modelo, no del punto exacto de la capital.** La API devuelve las coordenadas del punto que usó. La distancia respecto de la capital tiene mediana de **2.9 km** y máximo de **12.4 km** (Huancavelica). En departamentos de sierra, donde la lluvia cambia mucho con la altitud, la cifra puede no representar la ciudad. Además, por la documentación general de Open-Meteo (no por algo medido aquí), la API histórica entrega datos de modelos de reanálisis, no de estaciones meteorológicas.
4. **`dias_lluvia_fuerte` mide intensidad, no cantidad total.** 10 de los 25 departamentos tienen 0 días de 20 mm o más, y entre ellos hay lugares con mucha lluvia total, como Cerro de Pasco (774.0 mm, máximo diario 15.2 mm) o Ayacucho (485.1 mm). Una lluvia frecuente pero suave no llega al umbral.
5. **Lima (Huacho) 13.9 mm** con una sola ciudad costera: ya anotamos que no representa la sierra del departamento.

Nada de esto cambió ninguna cifra: la tabla y el CSV son exactamente lo que devolvió la API, procesado con la suma y el umbral de 20 mm.

In [18]:
# Guardar datos/lluvias_por_departamento.csv (ruta relativa, mismas filas y orden que la tabla de capitales)
columnas = ["departamento", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]
lluvias_por_departamento[columnas].to_csv("datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig")

leido = pd.read_csv("datos/lluvias_por_departamento.csv", encoding="utf-8-sig")
with open("datos/lluvias_por_departamento.csv", "rb") as f:
    inicio = f.read(3)
print("Archivo: datos/lluvias_por_departamento.csv ->", leido.shape, "| columnas:", leido.columns.tolist())
print("Empieza con BOM utf-8-sig:", inicio == b"\xef\xbb\xbf")
print("Departamentos únicos:", leido["departamento"].nunique(), "| valores nulos:", int(leido.isna().sum().sum()))
print("Igual a la tabla en memoria:", leido.round(5).equals(lluvias_por_departamento[columnas].round(5).reset_index(drop=True)))
print()
print(leido.to_string())

Archivo: datos/lluvias_por_departamento.csv -> (25, 6) | columnas: ['departamento', 'capital', 'latitud', 'longitud', 'lluvia_total_mm', 'dias_lluvia_fuerte']
Empieza con BOM utf-8-sig: True
Departamentos únicos: 25 | valores nulos: 0
Igual a la tabla en memoria: True

     departamento           capital   latitud  longitud  lluvia_total_mm  dias_lluvia_fuerte
0        Amazonas       Chachapoyas  -6.23169 -77.86903            436.5                   1
1          Áncash            Huaraz  -9.52614 -77.52869            975.3                   7
2        Apurímac           Abancay -13.63426 -72.88380            711.8                   2
3        Arequipa          Arequipa -16.39899 -71.53747            189.9                   0
4        Ayacucho          Ayacucho -13.16380 -74.22345            485.1                   0
5       Cajamarca         Cajamarca  -7.16378 -78.50027            773.3                   1
6           Cusco             Cusco -13.53188 -71.96701            710.7       